问题1：要沿着特定道路行走吗？
问题2：给shelter和neighorhood进行一样的正规化处理。
问题3：确定空间范围的获取和target的选择。


## CREATE A MODEL

In [1]:
import importlib.util
# Import pyflamegpu and some other libraries we will use in the tutorial
import pyflamegpu
import sys, random, math
import matplotlib.pyplot as plt
import pyflamegpu.codegen


In [ ]:
%env CUDA_PATH=D:\cuda12.9 

env: CUDA_PATH=D:\cuda12.9


In [ ]:
# Define the FLAME GPU model: 这个可以在后续的可视化窗口改名字
model = pyflamegpu.ModelDescription("Social_physical_shelter_Opt")

## Define Messages

In [ ]:

"""
  Location messages
"""  
message = model.newMessageBruteForce("student_agent_location_message")
message.newVariableID("id")
message.newVariableFloat("x")
message.newVariableFloat("y")

message = model.newMessageBruteForce("shelter_agent_location_message")
message.newVariableID("id")
message.newVariableFloat("x")
message.newVariableFloat("y")

stairwell_message = model.newMessageSpatial3D("location_stairwell")
stairwell_message.setMin(0, 0,0)
stairwell_message.setMax(500, 500, 500)
message.setRadius(200)
stairwell_message.newVariableID("id")
stairwell_message.newVariableFloat("class")


"""
  stairwell state messages 
"""
message = model.newMessageBruteForce("stairwell_congestion_message")
message.newVariableID("id")
message.newVariableFloat("congestion_state")

"""
  student agent state messages
"""
message = model.newMessageBruteForce("student_evacuate_state_message")
message.newVariableID("id")
message.newVariableInt("evacuate state")



## Agent definition

### student agent的变量

#### student agent 基础的变量

| Model Variable | Student Agent Variable Name | Description
| :--- | :--- | :--- |
| x | `x-location` | x |
| y | `y-location` | y |
| z | `z-location` | z |
| b_id | `building_id` | id of building where student agent live in  |
| p_id | `point_id` | student agent's ID  |

In [ ]:
student_agent = model.newAgent("student_agent")
student_agent.newVariableFloat("x")
student_agent.newVariableFloat("y")
student_agent.newVariableInt("building_id")
student_agent.newVariableInt("point_id")
student_agent.newVariableFloat("z")
student_agent.newVariableFloat("drift", 0)

#### 下面的栏用来讨论student agent还需要哪些变量

| Model Variable | Agent Variable Name | Description
| :--- | :--- | :--- |
| evacuate state | `evacuate_state` | 疏散状态 |
| ANIE | `Around_num_in_evacuation` | agent周围在疏散的人 |
| target_stairwell | `target_stairwell_id` | 要去的stairwell |
| target_shelter | `target_shelter_id` | 要去的shelter |

这里我需不需要把evacuate_state直接变成char，直接变成数字Int就好了。
| evacuate state | notation Int | 
| :--- | :--- | 
| focused | -1 |
| not evacuate | 0 |
| building evacuate | 1 |
| stairwell evacuate | 2 |
| neighborhood evacuate | 3 |

In [ ]:
student_agent.newVariableInt("evacuate_state")
student_agent.newVariableInt("ANIE")
student_agent.newVariableInt("target_stairwell_id")
student_agent.newVariableInt("target_shelter_id")


NameError: name 'model' is not defined

### stairwell agent 的变量

| Model Variable | Stairwell Agent Variable Name | Description
| :--- | :--- | :--- |
| $x$ | `x-location` | x |
| $y$ | `y-location` | y |
| $z$ | `z-location` | z |
| $s-id$ | `stairwell_id` | stairwell agent's ID  |

| stairwell state | notation Int | 
| :--- | :--- | 
| not crowded | 0 |
| crowded | 1 |

In [ ]:
stairwell_agent = model.newAgent("stairwell_agent")
stairwell_agent.newVariableFloat("x")
stairwell_agent.newVariableFloat("y")
stairwell_agent.newVariableInt("stairwell_id")
stairwell_agent.newVariableFloat("z")
stairwell_agent.newVariableInt("stairwell_state")

### agent behavior function 

#### student agent behavior function

student agent需要有哪些操作？（这里分为agent有意义的行为，和flamegpu需要的内置沟通message行为）

- 1.agent需要向flamegpu环境发送自己的位置信息，方便其他agent进行获取。（message）
- 2.agent需要获取其他agent的位置信息，从而根据阈值进行状态激活。
- 3.agent需要获取stairwell的位置信息，根据自己的激活状态，判断是否前往stairwell
- 4.agent要锁定前往的某个stairwell：这里需要根据熟悉度地图进行判断。（ 是内嵌还是在设置agent的时候设置呢？这里需要讨论 ）
- 5.agent需要向flamegpu环境发送自己是否处于stairwell。
- 6.处于stairwell中的agent需要根据stairwell的拥挤状态，来决定是否减速。或者说减速机制可以做得更加完善一点。
- 7.stairwell后前往某个shelter：这里需要根据熟悉度地图来判断。
- 8.向楼栋群发送消息。
- 9.有概率从楼栋群获取信息。从而改变自己对于疏散和选择的看法。


可用的变量类型：  https://docs.flamegpu.com/guide/creating-a-model/index.html#supported-types

In [ ]:
# student agent 发送位置信息以及疏散状态等信息：
@pyflamegpu.agent_function
def student_output_message(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    message_out.setVariableInt("point_id", pyflamegpu.getVariableInt("point_id"))
    message_out.setVariableInt("evacuate_state", pyflamegpu.getVariableInt("evacuate_state"))
    message_out.setLocation(
        pyflamegpu.getVariableFloat("x"),
        pyflamegpu.getVariableFloat("y"),
        pyflamegpu.getVariableFloat("z")
        )
    return pyflamegpu.ALIVE

In [ ]:
# student agent根据周围agent的数量来进行状态激活。
@pyflamegpu.agent_function
def student_agent_state_initial_shift(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    #获取student agent现在所处的evacuate state，如果evacuate处于not evacuate 或者 focused
    #那么就开始这个函数。
    evacuate_state = pyflamegpu.getVariableInt("evacuate_state")
    if evacuate_state <= 0:
        #遍历范围内agent的message？
        RADIUS = pyflamegpu.message_in.radius()
        # use a in-functon variable to store the count
        count = 0
        # Get this agent's x, y, z variables
        x1 = pyflamegpu.getVariableFloat("x")
        y1 = pyflamegpu.getVariableFloat("y")
        z1 = pyflamegpu.getVariableFloat("z")    
        # For each message in the message list which was output by a nearby agent
        for message in pyflamegpu.message_in(x1, y1, z1):
            x2 = message.getVariableFloat("x")
            y2 = message.getVariableFloat("y")
            z2 = message.getVariableFloat("z")
            # Calculate the distance to check the message is in range
            x21 = x2 - x1
            y21 = y2 - y1
            z21 = z2 - z1
            separation = math.sqrt(x21*x21 + y21*y21 + z21*z21)
            if separation < RADIUS and separation > 0 :
                count = count + 1
        #get threshold value from the environment
        if evacuate_state == -1:
            threshold = pyflamegpu.environment.getPropertyInt("focused_threshold")
        else:
            threshold = pyflamegpu.environment.getPropertyInt("not_evacuate_threshold")
        if count >= threshold:
            pyflamegpu.setVariableInt("evacuate_state", 1)
    return pyflamegpu.ALIVE



我现在已经初步选择了一个方案：

就是agent的变量里面初始化给一个熟悉度的值。

机制的耦合：
- student agent会选择最近的stairwell？
- student agent会跟随前面的人前往stairwell？
- student agent在慌乱中会前往最被常见使用的stairwell？（例如：宿舍楼中常被使用的是楼梯间而非消防通道，以至于一些人不会前往较近的stairwell）

关于调研：寻找stairwell的survey可以招募50个students。由论文表明这个人数是可行的。


In [ ]:
# student agent 获取flame gpu里的地图信息。这个地图信息由谁发布？需要另外加一些agent吗？所以把这个操作内嵌入到select function里面就好。
# 这个函数需要以及肯定要为选择stairwell和shelter服务的。

#message需要绑定stairwell发出的location messages。

# 1.选择最近的stairwell。
@pyflamegpu.agent_function
def student_select_stairwell(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageSpatial3D):
    # Get this agent's x, y, z variables
    x1 = pyflamegpu.getVariableFloat("x")
    y1 = pyflamegpu.getVariableFloat("y")
    z1 = pyflamegpu.getVariableFloat("z")
    
    # Initialize variables to store the nearest stairwell
    min_distance = float('inf')  
    nearest_x = 0.0
    nearest_y = 0.0
    nearest_z = 0.0
    nearest_stairwell_id = -2

    # For each message in the message list which was output by a nearby agent
    for message in pyflamegpu.message_in(x1, y1, z1):
        x2 = message.getVariableFloat("x")
        y2 = message.getVariableFloat("y")
        z2 = message.getVariableFloat("z")
        stairwell_id = message.getVariableInt("stairwell_id")
        
        # Calculate the distance to check the message is in range
        x21 = x2 - x1
        y21 = y2 - y1
        z21 = z2 - z1
        separation = math.sqrt(x21*x21 + y21*y21 + z21*z21)
        # Update nearest stairwell if this one is closer
        if separation < min_distance:
            min_distance = separation
            nearest_x = x2
            nearest_y = y2
            nearest_z = z2
            nearest_stairwell_id = stairwell_id

    # store these in agent variables
    pyflamegpu.setVariableFloat("nearest_stairwell_x", nearest_x)
    pyflamegpu.setVariableFloat("nearest_stairwell_y", nearest_y)
    pyflamegpu.setVariableFloat("nearest_stairwell_z", nearest_z)
    pyflamegpu.setVariableInt("nearest_stairwell_id", nearest_stairwell_id)

    return pyflamegpu.ALIVE

    
    

好的，我们完成了最基本的选择工作。

现在面临的问题是：

stairwell和student来源的message不一致呢。用一个函数？还是使用tpye？

会不会有那种跟随机制：

人们会有预设，希望前往最熟悉的stairwell。

比如说一开始人们很少找到最近的stairwell，但是只要有人找到了，那么后面的人就更有可能会跟随？


后续的工作：用强化学习的分布式训练，对不同agent实现其熟悉度学习的模拟。

#### stairwell agent behavior function

stairwell agent需要哪些操作？
- 1.需要向flamegpu发送自己的位置信息，便于student agent进行获取 （output）
- 2.获取student agent的信息，判断是否处于拥挤状态。（input）
- 3.向外界output自己的拥挤状态。

In [ ]:
@pyflamegpu.agent_function
def stairwell_output_message(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageBruteForce):
    message_out.setVariableInt("stairwell_id", pyflamegpu.getVariableInt("stairwell_id"))
    message_out.setVariableInt("stairwell_state", pyflamegpu.getVariableInt("stairwell_state"))
    message_out.setLocation(
        pyflamegpu.getVariableFloat("x"),
        pyflamegpu.getVariableFloat("y"),
        pyflamegpu.getVariableFloat("z")
        )
    return pyflamegpu.ALIVE    

In [ ]:
#stairwell设置拥挤状态
@pyflamegpu.agent_function
def stairwell_state(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    
    # Get this agent's x, y, z variables
    x1 = pyflamegpu.getVariableFloat("x")
    y1 = pyflamegpu.getVariableFloat("y")
    z1 = pyflamegpu.getVariableFloat("z")

    count = 0

    # For each message in the message list which was output by a nearby agent
    for message in pyflamegpu.message_in(x1, y1, z1):
        x2 = message.getVariableFloat("x")
        y2 = message.getVariableFloat("y")
        z2 = message.getVariableFloat("z")
        # Calculate the distance to check the message is in range
        
        if x1 == x2 and y1 == y2:
            count = count+1
    
    threshold_crowd_stairwell = pyflamegpu.environment.getPropertyInt("threshold_crowd_stairwell")
    if count >= threshold_crowd_stairwell:
        pyflamegpu.setVariableInt("threshold_crowd_stairwell", 1)
    
    return pyflamegpu.ALIVE

        
            # Process the message's variables e.g.
            # var = message.getVariableInt(...)    


### agent的状态变更


```python
m = pyflamegpu.ModelDescription("model")
# It contains an agent with 'variable 'x' and two states 'foo' and 'bar'
a = m.newAgent("agent")
a.newVariableInt("x")
a.newState("foo")
a.newState("bar")
```

如何设置只有处于focus或者evacuate状态的agent才能执行某种函数？
- 答案是：为执行函数设置进入函数和出去函数。
```python
af1 = a.newRTCFunction("example_function", ExampleFn_source)
af1.setInitialState("foo")
```


如何设置状态变更的函数？例如：设置agent没到楼梯口不会下楼
```python
#条件函数定义 (Python)
@pyflamegpu.agent_function_condition
def py_x_is_1() -> bool:
    return pyflamegpu.getVariableInt("x") == 1

#条件函数绑定
af1 = a.newRTCFunction("example_function", ExampleFn_source)
af1.setInitialState("foo")
af1.setEndState("bar")
x_is_1_translated = pyflamegpu.codegen.translate(x_is_1)
af1.setRTCFunctionCondition(x_is_1_translated)
```

这里可以设置为building_move函数。
函数的内容就包括进行基本移动以及判断是否移动到了那个点。

## Define Environmental Properties

In [ ]:
env = model.Environment()


In [ ]:
#这个是call back student agent的激活阈值。
env.newPropertyInt("focused_threshold" , 6)
env.newPropertyInt("not_evacuate_threshold", 3)
env.newPropertyInt("threshold_crowd_stairwell" , 50)

#activate_radium
env.newPropertyFloat("Range_be_activated", 15.0)
env.newPropertyFloat("Range_find_stairwell", 100.0)
env.newPropertyFloat("Range_find_shelter", 500.0)
env.newPropertyFloat("threshold_stairwell_congestion", 10.0)
    
    

我现在的问题呢，就是：message获取的范围，我要用message本身的好还是用函数遍历的好。我去查一查手册

ok。我现在懂了：
用message本身返回的应该是：交互半径中小格子的message，所以可能获取的范围会更大。但是我认为是可以接受的。

    for message in message_in(x1, y1):
        if message.getVariableUInt("id") != ID :
            x2 = message.getVariableFloat("x")
            y2 = message.getVariableFloat("y") 
            x21 = x2 - x1
            y21 = y2 - y1
            separation = math.sqrtf(x21*x21 + y21*y21)
            if separation < RADIUS and separation > 0 :

怎么获取建筑物的位置？也可以用macro env把建筑物的点放进去？好像不太ok的是，无法形成闭合的建筑物？那怎么放进去？已经解决了。

现在我需要解决一个问题：那就是device function可以使用其他的device function吗?测试结果不好，device function之间无法相互使用

### 熟悉度地图作为macro env property

现在我已经做了一个小测试，可以为我们的环境添加一个网格。agent可以快速地获取网格中的点。

然后我现在希望有一个文件可以快速输入。

可以的，非常方便，一列数变成矩阵就可以了

今天我实现了agent方便地获取熟悉度地图。熟悉度地图也做好了。


现在，差的是agent选择shelter的函数：要涵盖（拉取shelter的位置， 找到对应的熟悉度的值，根据自身对于距离和空间认知能力的考虑进行选择）

还有前往shelter的move函数。如何躲避building的位置，像行走一样。这个初步想用可见性图方法。
https://github.com/christopher-boustros/Unity-Visibility-Graph-Path-Planning-Simulation?tab=readme-ov-file#unity-visibility-graph-path-planning-simulation

## functions

## Function 的顺序很重要

但是我还有疑问：

不同的agent的函数能够并发吗？

不同agent的函数怎么绑定？

首先第一件事就是做一个简单的移动函数。

## 初始化population

我现在集齐了碎片：population的地理位置，状态，速度，熟悉度的差异。

熟悉度可以参考设置不同的策略

也可以设置各种问卷：例如你们会不会倾向于用手机联系同学。

小trick：
参考博弈论的code部分
- 可以为evacuees 的疏散状态设置为特征值
- 可以用随机的方法随机一个，然后添加到状态里面
~~~python
energy = max(
    random.normalvariate(INIT_ENERGY_MU, INIT_ENERGY_SIGMA), INIT_ENERGY_MIN
)
if MAX_ENERGY > 0.0:
    energy = min(energy, MAX_ENERGY)
instance.setVariableFloat("energy", energy)
~~~


困难是设置两种不同的agent

第一种方法：

先把create_agents函数嵌入到cudaSimulation里面去，再实例化。

cudaSimulation = pyflamegpu.CUDASimulation(model)

In [ ]:
class create_agents(pyflamegpu.HostFunction):
    def run(self, FLAMEGPU):
        Student_AGENT_COUNT = FLAMEGPU.environment.getPropertyUInt("Student_AGENT_COUNT") #总共有多少个学生agent

model.addInitFunction(create_agents())     

cudaSimulation = pyflamegpu.CUDASimulation(model)

NameError: name 'model' is not defined

第二种方法：
先实例化，再用setPopulationData的操作把agent的实际信息加入进去

In [ ]:
cudaSimulation = pyflamegpu.CUDASimulation(model)

random.seed(cudaSimulation.SimulationConfig().random_seed)

studentPopulation = pyflamegpu.AgentVector(model.Agent("prey"), num_student_agent)
for i in range(0, num_student_agent):
    Student_AGENT = studentPopulation[i]
    Student_AGENT.setVariableFloat("x", random.uniform(-1.0, 1.0))
    Student_AGENT.setVariableFloat("y", random.uniform(-1.0, 1.0))
    prey.setVariableFloat("vx", random.uniform(-1.0, 1.0))
    prey.setVariableFloat("vy", random.uniform(-1.0, 1.0))
    prey.setVariableFloat("steer_x", 0.0)
    prey.setVariableFloat("steer_y", 0.0) 
    prey.setVariableFloat("type", 1.0)
    prey.setVariableInt("life", random.randint(0, 50))

cudaSimulation.setPopulationData(predatorPopulation)

如何设置agent的行为逻辑，包含心理和社会要素

 PATH-U 的路径寻找（wayfinding）智能体（Qi Yang 2025）最新的一个wayfinding论文

 智能体根据目的地（AdestAdest​）、空间能力（AsbscolAsbscol​）、空间知识（AspkAspk​）和起点（AoriginAorigin​）生成，并从起点随机选择一个方向开始移动。

 检查是否到达目的地： 如果目的地可见（在视野列表 L f o v L fov ​ 中），智能体直接移动至目的地并终止。 
 
 探索阶段： 如果目的地不可见，智能体进入探索模式（explore），尝试通过空间知识或随机探索找到路径。 
 
 楼层判断： 智能体检查当前楼层（floorcurrent）是否与目标楼层（floortest）一致。 
 
 如果楼层不同，调用 FloorStrategy 处理跨楼层路径（例如使用电梯或楼梯）。 
 
 不确定性计算与局部策略： 计算当前路径的不确定性（ U w f U wf ​ ），基于当前位置、目的地、视野、记忆和空间能力。 
 
 调用 LocalStrategy（分为 L1 和 L2 模式）选择下一步移动的节点（ N s u b N o d e N subNode ​ ）。
 
  L1：直接选择可见的路径节点。 L2：从可用路径列表（ R R）中选择最优节点。 决策点处理： 如果当前位置是决策点（ N d p N dp ​ 中的节点），检查是否有指向目的地的标识（helpfulSign）。若有，直接移动至目标节点并更新记忆（ M M）。
  
   移动与记忆更新： 移动到子节点（ N s u b N o d e N subNode ​ ），并更新短期记忆缓冲区（ M M）以记录已访问的节点。 终止条件： 当智能体到达目的地（ A p o s = A d e s t A pos ​ =A dest ​ ）时，过程结束。

但是上述的path-u 基本没有考虑到社区内熟悉度，也没有考虑到拥挤和前方人员的影响。

如果要考虑到避免碰撞和实现寻路算法的话，打格子还是必要的，怎么将网格嵌入到3d的空间里面也很重要。